# Adversarial Arms Race -- Multi-Agent Swarm
**Run this on Lightning AI A100 (40GB)**

- 5 specialized attacker agents (email / chat / tool_output / document / code) on Gemma 3 1B
- 1 defender (Gemma 3 4B) that trains online
- UCB coordinator + distributed multi-source attacks

In [ ]:
# 1. Install deps
!pip install -q -U \
    bitsandbytes>=0.46.1 \
    peft>=0.12.0 \
    transformers>=4.47.0 \
    accelerate>=0.34.0 \
    rich

In [ ]:
# 2. Clone / update repo
import os

REPO = "https://github.com/nileshpatil6/promptinject-env-agents.git"
WORKDIR = "/teamspace/studios/this_studio/promptinject-env-agents"

if os.path.exists(WORKDIR):
    !cd {WORKDIR} && git pull
else:
    !git clone {REPO} {WORKDIR}

os.chdir(WORKDIR)
print("Working dir:", os.getcwd())

In [ ]:
# 3. Set HuggingFace token
import os
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"   # <-- paste your token

In [ ]:
# 4. Verify GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# 5. Run the arms race
# A100 settings: group=8, episodes=30, rounds=6
# Logs saved to arena/checkpoints_multi/arms_race_multi_log.jsonl

!python arena/run_multi.py \
    --no-viz \
    --episodes 30 \
    --rounds 6 \
    --group 8 \
    --save-dir arena/checkpoints_multi \
    --save-every 5

In [ ]:
# 6. Plot evasion curves from log
import json, matplotlib.pyplot as plt

log_path = "arena/checkpoints_multi/arms_race_multi_log.jsonl"

episodes, atk_evasion, def_accuracy = [], [], []
vec_data = {}

with open(log_path) as f:
    for line in f:
        d = json.loads(line)
        if d["type"] == "episode":
            episodes.append(d["episode"])
            atk_evasion.append(d["atk_evasion"])
            def_accuracy.append(d["def_accuracy"])
            for v, n in d.get("vector_hall_of_fame", {}).items():
                vec_data.setdefault(v, []).append(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(episodes, atk_evasion, label="ATK evasion", color="red", linewidth=2)
axes[0].plot(episodes, def_accuracy, label="DEF accuracy", color="blue", linewidth=2)
axes[0].set_title("Arms Race Dynamics")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Rate")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for v, vals in vec_data.items():
    eps = episodes[:len(vals)]
    axes[1].plot(eps, vals, label=v, linewidth=1.5)
axes[1].set_title("Hall of Fame Size per Vector")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Entries")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("arena/checkpoints_multi/arms_race_curves.png", dpi=150)
plt.show()
print(f"Final ATK evasion: {atk_evasion[-1]:.1%}  |  DEF accuracy: {def_accuracy[-1]:.1%}")

In [ ]:
# 7. Per-vector evasion breakdown (last episode)
import json

log_path = "arena/checkpoints_multi/arms_race_multi_log.jsonl"
last_ep = None

with open(log_path) as f:
    for line in f:
        d = json.loads(line)
        if d["type"] == "episode":
            last_ep = d

if last_ep:
    print(f"Episode {last_ep['episode']} final stats")
    print(f"  Overall ATK evasion : {last_ep['atk_evasion']:.1%}")
    print(f"  Overall DEF accuracy: {last_ep['def_accuracy']:.1%}")
    print()
    rates = last_ep.get("coordinator_summary", {}).get("agent_evasion_rates", {})
    print("  Per-vector evasion rates:")
    for v, r in sorted(rates.items(), key=lambda x: -x[1]):
        bar = "#" * int(r * 30) + "." * (30 - int(r * 30))
        print(f"    {v:12s} [{bar}] {r:.0%}")